### **Download and import libs**

In [ ]:
pip install opfython

In [ ]:
from sklearn import svm
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score, ShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.image import extract_patches_2d
from opfython.models import SupervisedOPF
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import glob
import os
from zipfile import ZipFile
import cv2
import numpy as np
import time
import pandas as pd
import torch
from torchvision import transforms
import torchvision.transforms.functional as F

### **Importing .csv data, scaling and splitting**

In [ ]:
spiral_dataset = pd.read_csv('Spiral_HandPD.csv').drop(['368', '2', 'Unnamed: 11'], axis=1)
meander_dataset = pd.read_csv('Meander_HandPD.csv').drop(['368', '2', 'Unnamed: 11'], axis=1)
spiral_dataset = spiral_dataset.rename(columns={'9':'class_type', 'Unnamed: 3': 'RMS', 'Unnamed: 4': 'MAX_HT_ET', 'Unnamed: 6': 'MIN_HT_ET', 'Unnamed: 5': 'STD_DEV_HT_ET', 'Unnamed: 7': 'MRT', 'Unnamed: 8': 'MAX_ET', 'Unnamed: 9': 'MIN_ET', 'Unnamed: 10': 'STD_ET'})
spiral_dataset.head()

,class_type,RMS,MAX_HT_ET,STD_DEV_HT_ET,MIN_HT_ET,MRT,MAX_ET,MIN_ET,STD_ET
0,1,3521.258301,6247.052734,30801.992188,0.014133,26.785328,176.600113,0.002130,1781.795898
1,1,4098.876465,6032.535156,34369.703125,0.022838,26.529615,168.352737,0.084960,1443.217529
2,1,3854.601807,6453.114746,34709.445312,0.000251,23.670755,180.898300,0.009303,1621.750000
3,1,4069.221924,6844.231445,32181.263672,0.000168,23.456329,179.116043,0.021419,1454.390137
4,1,4104.271973,6949.925293,36444.953125,0.004731,22.488258,188.256210,0.000000,1553.536499


In [ ]:
spiral_dataset.columns

Index(['class_type', 'RMS', 'MAX_HT_ET', 'STD_DEV_HT_ET', 'MIN_HT_ET', 'MRT',
       'MAX_ET', 'MIN_ET', 'STD_ET'],
      dtype='object')

In [ ]:
spiral_dataset

,class_type,RMS,MAX_HT_ET,STD_DEV_HT_ET,MIN_HT_ET,MRT,MAX_ET,MIN_ET,STD_ET
0,1,3521.258301,6247.052734,30801.992188,0.014133,26.785328,176.600113,0.002130,1781.795898
1,1,4098.876465,6032.535156,34369.703125,0.022838,26.529615,168.352737,0.084960,1443.217529
2,1,3854.601807,6453.114746,34709.445312,0.000251,23.670755,180.898300,0.009303,1621.750000
3,1,4069.221924,6844.231445,32181.263672,0.000168,23.456329,179.116043,0.021419,1454.390137
4,1,4104.271973,6949.925293,36444.953125,0.004731,22.488258,188.256210,0.000000,1553.536499
...,...,...,...,...,...,...,...,...,...
363,2,5593.215820,7997.703613,36371.226562,0.000190,26.964298,179.071930,0.059952,1885.214478
364,2,7986.296387,7497.133789,29909.390625,2.613612,22.278151,203.759109,0.033372,1667.547974
365,2,4652.542969,7275.474121,39475.414062,0.240567,21.622019,198.719940,0.000000,1469.676147
366,2,5183.951172,8040.058594,35332.328125,0.020707,27.244106,182.095047,0.018142,1784.178101


In [ ]:
meander_dataset = meander_dataset.rename(columns={'9':'class_type', 'Unnamed: 3': 'RMS', 'Unnamed: 4': 'MAX_HT_ET', 'Unnamed: 6': 'MIN_HT_ET', 'Unnamed: 5': 'STD_DEV_HT_ET', 'Unnamed: 7': 'MRT', 'Unnamed: 8': 'MAX_ET', 'Unnamed: 9': 'MIN_ET', 'Unnamed: 10': 'STD_ET'})
meander_dataset.head()

,class_type,RMS,MAX_HT_ET,STD_DEV_HT_ET,MIN_HT_ET,MRT,MAX_ET,MIN_ET,STD_ET
0,1,3176.216064,7098.378906,46569.035156,0.000672,21.280848,224.197754,0.156795,802.821106
1,1,2751.015869,6263.803711,44059.597656,0.000000,22.056967,212.937836,0.009198,939.975647
2,1,3050.623779,6548.623047,40298.109375,0.000026,22.451719,223.401764,0.000000,997.580139
3,1,2594.598877,6989.159180,54217.632812,0.000000,30.559263,233.222504,0.367697,2060.858887
4,1,3310.786865,6060.232422,35212.757812,0.000667,18.138407,196.811325,0.073079,562.886475


In [ ]:
def only_scale_samples(dataset):
  samples, labels = dataset.iloc[:,1:].values, dataset.iloc[:,0].values
  scaler = StandardScaler()
  scaled_samples = scaler.fit_transform(X=samples,y=labels)
  return scaled_samples, labels

In [ ]:
def scale_and_split_samples(dataset, test_ratio):
  samples, labels = dataset.iloc[:,1:].values, dataset.iloc[:,0].values
  scaler = StandardScaler()
  scaled_samples = scaler.fit_transform(X=samples,y=labels)
  training_samples, test_samples, training_labels, test_labels = train_test_split(scaled_samples, labels, test_size = test_ratio)
  return training_samples, test_samples, training_labels, test_labels

In [ ]:
meander_training_samples, meander_test_samples, meander_training_labels, meander_test_labels = scale_and_split_samples(meander_dataset, 0.25)

In [ ]:
spiral_training_samples, spiral_test_samples, spiral_training_labels, spiral_test_labels = scale_and_split_samples(spiral_dataset, 0.25)

In [ ]:
meander_samples, meander_labels = only_scale_samples(meander_dataset)
spiral_samples, spiral_labels = only_scale_samples(spiral_dataset)

In [ ]:
def merge_scale_and_split_samples(dataset, test_ratio):
  samples, labels = dataset.iloc[:,1:].values, dataset.iloc[:,0].values

  training_samples, test_samples, training_labels, test_labels = train_test_split(samples, labels, test_size = test_ratio)
  scaler = StandardScaler()
  scaled_training_samples = scaler.fit_transform(X=training_samples,y=training_labels)
  scaled_test_samples = scaler.fit_transform(X=test_samples, y=test_labels)

  return scaled_training_samples, scaled_test_samples, training_labels, test_labels

In [ ]:
merged_parkinson_dataset = pd.concat([meander_dataset, spiral_dataset], ignore_index=True)
merged_parkinson_dataset.head()

,class_type,RMS,MAX_HT_ET,STD_DEV_HT_ET,MIN_HT_ET,MRT,MAX_ET,MIN_ET,STD_ET
0,1,3176.216064,7098.378906,46569.035156,0.000672,21.280848,224.197754,0.156795,802.821106
1,1,2751.015869,6263.803711,44059.597656,0.000000,22.056967,212.937836,0.009198,939.975647
2,1,3050.623779,6548.623047,40298.109375,0.000026,22.451719,223.401764,0.000000,997.580139
3,1,2594.598877,6989.159180,54217.632812,0.000000,30.559263,233.222504,0.367697,2060.858887
4,1,3310.786865,6060.232422,35212.757812,0.000667,18.138407,196.811325,0.073079,562.886475


In [ ]:
parkinson_training_samples, parkinson_test_samples, parkinson_training_labels, parkinson_test_labels = merge_scale_and_split_samples(merged_parkinson_dataset, 0.25)

### **Train models**

In [ ]:
def class_accuracy(predicted_labels, true_labels):
  cm = confusion_matrix(predicted_labels, true_labels)
  true_negative, false_positive = cm[0]
  false_negative, true_positive = cm[1]

  control_accuracy = true_negative / (true_negative + false_positive)
  patient_accuracy = true_positive / (true_positive + false_negative)

  return control_accuracy, patient_accuracy

In [ ]:
def train_test_models_no_cv(training_samples, training_labels, test_samples, test_labels):
  # Naive Bayes

  meander_gaussian_no_cv = GaussianNB()
  predicted_meander_bayes_no_cv = meander_gaussian_no_cv.fit(training_samples, training_labels).predict(test_samples)
  meander_naive_bayes_acc = accuracy_score(test_labels, predicted_meander_bayes_no_cv)
  print(f"Number of mislabeled points out of a total {test_samples.shape[0]} points : {(test_labels != predicted_meander_bayes_no_cv).sum()}")
  print(f"Global Naive Bayes Accuracy Score w/o Crossval: {meander_naive_bayes_acc}")
  naive_bayes_control_acc, naive_bayes_patient_acc = class_accuracy(predicted_meander_bayes_no_cv, test_labels)

  print(f"Control class Naive Bayes Accuracy w/o Crossval: {naive_bayes_control_acc}")
  print(f"Pacient class Naive Bayes Accuracy w/o Crossval: {naive_bayes_patient_acc}\n")

  # Optimum Path Forest

  meander_opf_no_cv = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
  meander_opf_no_cv.fit(training_samples, training_labels)
  predicted_meander_labels = meander_opf_no_cv.predict(test_samples)
  meander_opf_acc = accuracy_score(test_labels, predicted_meander_labels)
  print(f"Global OPF Accuracy Score w/o no Crossval: {meander_opf_acc}")
  opf_control_acc, opf_patient_acc = class_accuracy(predicted_meander_labels, test_labels)
  print(f"Control class OPF Accuracy w/o Crossval: {opf_control_acc}")
  print(f"Pacient class OPF Accuracy w/o Crossval: {opf_patient_acc}\n")

  # Support Vector Machine

  meander_svm_no_cv = svm.SVC()
  meander_svm_predicted = meander_svm_no_cv.fit(training_samples, training_labels).predict(test_samples)
  meander_svm_acc_no_cv = accuracy_score(test_labels, meander_svm_predicted)
  print(f"Global SVM Accuracy Score w/o no crossval: {meander_svm_acc_no_cv}")
  svm_control_acc, svm_patient_acc = class_accuracy(meander_svm_predicted, test_labels)
  print(f"Control class SVM Accuracy w/o Crossval: {svm_control_acc}")
  print(f"Pacient class SVM Accuracy w/o Crossval: {svm_patient_acc}")

In [ ]:
def crossval_models(samples, labels, test_ratio):
  n_samples = samples.shape[0]
  cv = ShuffleSplit(n_splits=5, test_size=test_ratio, random_state=0)
  naive_bayes_cv = GaussianNB()
  opf_cv = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
  svm_cv = svm.SVC()
  selected_models = [naive_bayes_cv, svm_cv]
  for model in selected_models:
    scores = cross_val_score(model, samples, labels, cv=cv)
    print(model)
    print(f"Model {model}: {scores.mean()} Accuracy with a standard deviation of {scores.std()}")


In [ ]:
naive_bayes_cv = GaussianNB()
print(naive_bayes_cv)

GaussianNB()


#### **Experiment #01**

In [ ]:
train_test_models_no_cv(meander_training_samples, meander_training_labels, meander_test_samples, meander_test_labels)

Number of mislabeled points out of a total 92 points : 45
Global Naive Bayes Accuracy Score w/o Crossval: 0.5108695652173914
Control class Naive Bayes Accuracy w/o Crossval: 0.22641509433962265
Pacient class Naive Bayes Accuracy w/o Crossval: 0.8974358974358975

2025-12-02 02:14:38,938 - opfython.models.supervised — INFO — Overriding class: OPF -> SupervisedOPF.
2025-12-02 02:14:38,939 - opfython.core.opf — INFO — Creating class: OPF.
2025-12-02 02:14:38,948 - opfython.core.opf — DEBUG — Distance: log_squared_euclidean | Pre-computed distance: False.
2025-12-02 02:14:38,954 - opfython.core.opf — INFO — Class created.
2025-12-02 02:14:38,963 - opfython.models.supervised — INFO — Class overrided.
2025-12-02 02:14:38,964 - opfython.models.supervised — INFO — Fitting classifier ...
2025-12-02 02:14:38,973 - opfython.models.supervised — DEBUG — Finding prototypes ...
2025-12-02 02:14:42,328 - opfython.models.supervised — DEBUG — Prototypes: [234, 267, 191, 61, 18, 98, 137, 75, 228, 54, 180,

In [ ]:
train_test_models_no_cv(spiral_training_samples, spiral_training_labels, spiral_test_samples, spiral_test_labels)

Number of mislabeled points out of a total 92 points : 28
Global Naive Bayes Accuracy Score w/o Crossval: 0.6956521739130435
Control class Naive Bayes Accuracy w/o Crossval: 0.4
Pacient class Naive Bayes Accuracy w/o Crossval: 0.8387096774193549

2025-12-02 02:14:42,662 - opfython.models.supervised — INFO — Overriding class: OPF -> SupervisedOPF.
2025-12-02 02:14:42,663 - opfython.core.opf — INFO — Creating class: OPF.
2025-12-02 02:14:42,665 - opfython.core.opf — DEBUG — Distance: log_squared_euclidean | Pre-computed distance: False.
2025-12-02 02:14:42,667 - opfython.core.opf — INFO — Class created.
2025-12-02 02:14:42,667 - opfython.models.supervised — INFO — Class overrided.
2025-12-02 02:14:42,669 - opfython.models.supervised — INFO — Fitting classifier ...
2025-12-02 02:14:42,673 - opfython.models.supervised — DEBUG — Finding prototypes ...
2025-12-02 02:14:42,759 - opfython.models.supervised — DEBUG — Prototypes: [51, 0, 235, 139, 205, 108, 191, 109, 105, 66, 96, 185, 159, 180, 

/tmp/ipython-input-847234237.py:6: RuntimeWarning: invalid value encountered in scalar divide
  control_accuracy = true_negative / (true_negative + false_positive)


### **Experiment #02**

In [ ]:
crossval_models(meander_samples, meander_labels, 0.25)

2025-12-02 02:14:43,142 - opfython.models.supervised — INFO — Overriding class: OPF -> SupervisedOPF.
2025-12-02 02:14:43,145 - opfython.core.opf — INFO — Creating class: OPF.
2025-12-02 02:14:43,146 - opfython.core.opf — DEBUG — Distance: log_squared_euclidean | Pre-computed distance: False.
2025-12-02 02:14:43,151 - opfython.core.opf — INFO — Class created.
2025-12-02 02:14:43,153 - opfython.models.supervised — INFO — Class overrided.
GaussianNB()
Model GaussianNB(): 0.5347826086956522 Accuracy with a standard deviation of 0.12739204629550965
SVC()
Model SVC(): 0.7891304347826087 Accuracy with a standard deviation of 0.04591024365639756


### **Experiment #03**

In [ ]:
train_test_models_no_cv(parkinson_training_samples, parkinson_training_labels, parkinson_test_samples, parkinson_test_labels)

Number of mislabeled points out of a total 184 points : 46
Global Naive Bayes Accuracy Score w/o Crossval: 0.75
Control class Naive Bayes Accuracy w/o Crossval: 0.07142857142857142
Pacient class Naive Bayes Accuracy w/o Crossval: 0.8058823529411765

2025-12-02 02:14:43,300 - opfython.models.supervised — INFO — Overriding class: OPF -> SupervisedOPF.
2025-12-02 02:14:43,303 - opfython.core.opf — INFO — Creating class: OPF.
2025-12-02 02:14:43,305 - opfython.core.opf — DEBUG — Distance: log_squared_euclidean | Pre-computed distance: False.
2025-12-02 02:14:43,308 - opfython.core.opf — INFO — Class created.
2025-12-02 02:14:43,312 - opfython.models.supervised — INFO — Class overrided.
2025-12-02 02:14:43,316 - opfython.models.supervised — INFO — Fitting classifier ...
2025-12-02 02:14:43,333 - opfython.models.supervised — DEBUG — Finding prototypes ...
2025-12-02 02:14:43,699 - opfython.models.supervised — DEBUG — Prototypes: [514, 471, 77, 228, 251, 124, 59, 376, 467, 551, 294, 276, 86, 